# KTSDE 논문 재실행 (Kaggle)

## ⚠️ 셀을 하나씩 누르지 말 것 — **Save Version → Save & Run All**

우측 상단 **Save Version** → **Save & Run All (Commit)** → Save. 브라우저를 닫아도 된다.
끝나면 노트북 **Output** 탭에 결과가 영구 보존된다.

**인터랙티브 세션(`Shift+Enter`)으로 2시간을 돌리면 자리를 비운 사이 세션이 끝나고
`/kaggle/working` 이 비워진다** — 2026-10-19 에 실제로 그렇게 체크포인트 10개를 잃었다.
Input·GPU·Internet 설정은 커밋 실행에 그대로 따라간다.

셀은 3개이고 고칠 것은 없다.

### 먼저 우측 패널에서 3가지

| 설정 | 값 |
|---|---|
| **Accelerator** | `GPU T4 x2` (또는 P100) |
| **Internet** | `On` |
| **Input** → Add Input | 업로드한 `train_bundle.zip` 데이터셋 |

> 회색으로 안 눌리면 휴대폰 인증이 필요하다 (Settings → Phone Verification).
> 데이터셋에 New Version 을 올렸으면 **Input 을 제거하고 다시 추가**해야 한다 —
> 안 하면 옛 버전에 고정된다.

### 걸리는 시간

셀 1 약 1분 · **셀 2 약 2시간** (제안 모형 5시드 25분 + TFT 5시드 1시간 30분 + 집계 10분) · 셀 3 1분.

시간이 부족하면 셀 2 맨 위의 `SEEDS` 를 `[42, 43, 44]` 로 줄인다.

## 셀 1 — 준비와 점검

여기서 멈추면 아래를 돌려도 의미가 없다. 메시지대로 고치고 다시 실행.

In [ ]:
import glob, os, shutil, subprocess, zipfile

import torch
import yaml

REPO = 'https://github.com/sungjunpk/Capstone_Stock_Price_Prediction.git'
FREEZE = 'docs/freeze_panel_20261018.json'

# ── 1) GPU
if not torch.cuda.is_available():
    raise SystemExit('❌ GPU 가 없다 — 우측 Accelerator 를 GPU 로 바꾸고 Run → Restart Session')
print('✅ GPU:', torch.cuda.get_device_name(0))

# ── 2) 코드: 항상 새로 받는다 (낡은 클론으로 도는 사고를 막는다)
os.chdir('/kaggle/working')
shutil.rmtree('Capstone_Stock_Price_Prediction', ignore_errors=True)
subprocess.run(['git', 'clone', '-q', REPO], check=True)
os.chdir('/kaggle/working/Capstone_Stock_Price_Prediction')
head = subprocess.run(['git', 'log', '-1', '--format=%h %s'],
                      capture_output=True, text=True).stdout.strip()
print('✅ 코드:', head)
if not os.path.exists(FREEZE):
    raise SystemExit(f'❌ {FREEZE} 이 없다 — 로컬에서 git push 를 먼저 할 것')

# ── 3) 데이터: Input 에서 찾아 붙인다
os.makedirs('data/processed', exist_ok=True)
parqs = glob.glob('/kaggle/input/**/panel*.parquet', recursive=True)
zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
if parqs:
    for f in glob.glob(f'{os.path.dirname(parqs[0])}/*.parquet'):
        shutil.copy(f, 'data/processed/')
elif zips:
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall('.')
else:
    raise SystemExit('❌ 데이터를 못 찾았다 — 우측 Add Input 으로 데이터셋을 붙일 것')
if not os.path.exists('data/processed/panel.parquet'):
    raise SystemExit('❌ panel.parquet 이 없다. 접미사 붙은 묶음을 올렸다 — '
                     'package_data.py 를 프로파일 없이 다시 돌려 만든 zip 을 쓸 것\n'
                     f"   있는 파일: {sorted(os.listdir('data/processed'))}")
print('✅ 데이터:', sorted(os.listdir('data/processed')))

# ── 4) 설정이 논문 설정인지
enc = yaml.safe_load(open('configs/config.yaml'))['model']['encoder']
if (enc['d_model'], enc['n_layers']) != (32, 1):
    raise SystemExit(f"❌ 논문 설정이 아니다 (d_model={enc['d_model']}, n_layers={enc['n_layers']})")
print(f"✅ 설정: d_model={enc['d_model']} n_layers={enc['n_layers']} dropout={enc['dropout']}")

# ── 5) 자료가 로컬 동결 기록과 같은지 ★
print('\n── 자료 동결 대조 ' + '─' * 40)
if subprocess.run(f'python scripts/freeze_panel.py --verify {FREEZE}', shell=True).returncode:
    raise SystemExit('\n❌ 자료가 로컬 동결 기록과 다르다.\n'
                     '   → 캐글 우측 Input 에서 데이터셋을 제거하고 다시 추가할 것'
                     ' (New Version 이 자동 반영되지 않는다)')
print('\n✅ 준비 끝 — 셀 2 를 실행한다')

## 셀 2 — 학습과 집계 (약 2시간)

중간에 끊기면 **이 셀만 다시 실행**하면 된다. 이미 끝난 시드는 건너뛴다
(셀 1 을 다시 돌리면 체크포인트가 지워지니 그러지 말 것).

마지막에 나오는 표가 논문 〈Table 4〉가 될 수치다.

In [ ]:
import glob, os, re, subprocess, time

SEEDS = [42, 43, 44, 45, 46]      # 시간이 부족하면 [42, 43, 44]


def run(cmd):
    t0 = time.time()
    subprocess.run(cmd, shell=True, check=True)
    return (time.time() - t0) / 60


# 해당 시드 체크포인트가 이미 있으면 경로를 준다 — 재실행 때 건너뛰기용
def ckpt(seed, tft):
    for p in sorted(glob.glob(f'outputs/checkpoints/phase1_*_seed{seed}.pt')):
        n = os.path.basename(p)
        if 'smoke' in n or (('_tft_' in n) != tft):
            continue
        return p
    return None


# ── 배관 점검
print('── 배관 점검 ' + '─' * 44, flush=True)
run('python scripts/train.py --smoke')

# ── 제안 모형 / TFT 원본, 시드마다
for tft, label, cmd in ((False, 'ours', 'python scripts/train.py --seed {s}'),
                        (True, 'tft', 'python scripts/arch_compare.py --train tft --seed {s}')):
    for s in SEEDS:
        if ckpt(s, tft):
            print(f'\n── {label} seed {s}: 이미 있다 — 건너뛴다', flush=True)
            continue
        print(f'\n── {label} seed {s} ' + '─' * 44, flush=True)
        print(f'   {label} seed {s}: {run(cmd.format(s=s)):.1f}분', flush=True)

# ── 체크포인트 이름 앞부분을 실제 파일에서 찾는다 (config 해시가 바뀌어도 돈다)
A = re.sub(r'_seed\d+\.pt$', '', os.path.basename(ckpt(SEEDS[0], False)))
B = re.sub(r'_seed\d+\.pt$', '', os.path.basename(ckpt(SEEDS[0], True)))
print(f'\n제안 모형 {A} / TFT 원본 {B}')

# ── 〈Table 4〉 집계 ★ 이 리포트가 표의 출처 파일이다
print('\n── Table 4 집계 ' + '─' * 41, flush=True)
run(f'python scripts/seed_aggregate.py --a {A} --b {B} '
    f"--seeds {' '.join(map(str, SEEDS))} --test --freeze docs/freeze_panel_20261018.json")

# ── 시드별 DM 검정 (추론만)
print('\n── 시드별 DM 검정 ' + '─' * 39, flush=True)
for s in SEEDS:
    print(f'\n[seed {s}]', flush=True)
    run(f'python scripts/dm_test.py --a {ckpt(s, False)} --b {ckpt(s, True)}')

# ── 리포트 전문을 로그에 찍는다. 세션이 비워져도 로그는 남으므로 이것이 마지막 백업이다
import glob
import json as _json

latest = sorted(glob.glob('outputs/reports/seed_aggregate_*.json'))[-1]
print('\n── 집계 리포트 전문 (' + latest + ') ' + '─' * 20, flush=True)
print(_json.dumps(_json.load(open(latest)), ensure_ascii=False, indent=1), flush=True)

print('\n✅ 끝 — 셀 3 으로 결과를 내려받는다', flush=True)

## 셀 3 — 결과 내려받기

우측 **Output** 패널의 `ktsde_rerun.zip` 을 받아서 로컬 `outputs/` 에 푼다.

> 밤새 돌리고 왔다면 커널이 재시작돼 `cwd` 가 `/kaggle/working` 으로 돌아가 있다.
> 이 셀은 저장소 폴더를 직접 찾으므로 **이 셀만 다시 실행하면 된다**.


In [ ]:
import glob, os, shutil

# 커널이 재시작되면 cwd 가 /kaggle/working 으로 돌아간다 — 저장소를 직접 찾는다
ckpts = sorted(glob.glob('/kaggle/working/**/outputs/checkpoints/*.pt', recursive=True))
reports = sorted(glob.glob('/kaggle/working/**/outputs/reports/*.json', recursive=True))
print(f'체크포인트 {len(ckpts)}개 / 리포트 {len(reports)}개')

if not ckpts and not reports:
    raise SystemExit('❌ 결과가 없다 — 세션이 끊겨 /kaggle/working 이 비었다. '
                     '셀 1 부터 다시 돌려야 한다.')

root = (ckpts or reports)[0].split('/outputs/')[0]
print('저장소:', root)

dst = '/kaggle/working/download'
shutil.rmtree(dst, ignore_errors=True)
os.makedirs(dst, exist_ok=True)
for sub in ('checkpoints', 'reports'):
    src = f'{root}/outputs/{sub}'
    if os.path.isdir(src):
        shutil.copytree(src, f'{dst}/{sub}')
        print(f'  {sub}: {len(os.listdir(src))}개')

shutil.make_archive('/kaggle/working/ktsde_rerun', 'zip', dst)
size = os.path.getsize('/kaggle/working/ktsde_rerun.zip') / 1e6
print(f'→ 우측 Output 패널에서 ktsde_rerun.zip 다운로드 ({size:.1f} MB)')
print('   로컬에서: unzip -o ~/Downloads/ktsde_rerun.zip -d outputs/')

---
## 막히면

| 메시지 | 해결 |
|---|---|
| `GPU 가 없다` | 우측 Accelerator → GPU → Run → Restart Session |
| `freeze_panel_20261018.json 이 없다` | 로컬에서 `git push` 를 안 했다 |
| `데이터를 못 찾았다` | 우측 **Add Input** 으로 데이터셋을 붙인다 |
| `panel.parquet 이 없다` | 프로파일 붙은 묶음을 올렸다. `package_data.py` 를 인자 없이 다시 |
| `자료가 로컬 동결 기록과 다르다` | Input 데이터셋이 옛 버전. 제거 후 다시 추가 |
| `git clone` 이 멈춤 | Internet: On |
| `CUDA out of memory` | 셀 2 의 두 학습 명령에 `--batch-size 128` 을 붙인다 |
| 세션이 끊김 | `/kaggle/working` 이 비므로 **셀 1 부터** 다시 |
| 셀 2 만 실패 | 셀 2 만 다시 실행하면 끝난 시드는 건너뛴다 |
| 셀 3 에서 `No such file or directory: .../download` | 커널이 재시작된 것. 셀 3 을 다시 실행한다 (저장소를 직접 찾는다) |